In [2]:
%%writefile phasenet_common.py

import os, random
import numpy as np
import cv2
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# ============ CONFIG - must match the sonification run being evaluated ============
SR = 22050
N_FFT = 4096                 # = n_fft_out used for your dataset conversion
HOP = N_FFT // 2
WIN = N_FFT
TARGET_LOW, TARGET_HIGH = 1000.0, 4000.0
IMG_H, IMG_W = 513, 128      # source image resolution (unchanged I-GLA preprocessing)
IMG_ROOT = r"D:\Documents\Iquisitionis\103\vehicle-type-recognition\Dataset"
CACHE_PATH = "phasenet_mags.npz"   # delete this file if you change N_FFT or IMG_ROOT
SEED = 42
TEST_FRAC = 0.1
IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff')
# ===================================================================================

N_BINS = N_FFT // 2 + 1
_freqs = np.linspace(0, SR / 2, N_BINS)
BAND_IDX = np.where((_freqs >= TARGET_LOW) & (_freqs <= TARGET_HIGH))[0]
BAND_LO, BAND_HI = int(BAND_IDX[0]), int(BAND_IDX[-1]) + 1   # in-band rows = [BAND_LO:BAND_HI]
N_BAND = BAND_HI - BAND_LO                                     # 558 at N_FFT=4096
PAD_H = N_BAND + (-N_BAND) % 16                                # U-Net needs a multiple of 16
AUDIO_LEN = HOP * (IMG_W - 1)


# ---- unchanged from your pipeline ----
def frequency_remap_hires(spectrogram, sr=22050, n_fft_out=4096, target_low=1000.0, target_high=4000.0):
    n_src, n_frames = spectrogram.shape
    n_bins_out = n_fft_out // 2 + 1
    freq_axis_src = np.linspace(0, sr / 2, n_src)
    b = target_low
    a = (target_high - target_low) / np.log(freq_axis_src[-1] + 1)
    f_new = a * np.log(freq_axis_src + 1) + b
    dst_pos = f_new / (sr / n_fft_out)
    boundaries = (dst_pos[:-1] + dst_pos[1:]) / 2.0
    remapped = np.zeros((n_bins_out, n_frames), dtype=spectrogram.dtype)
    freq_axis_out = np.linspace(0, sr / 2, n_bins_out)
    in_band = (freq_axis_out >= target_low) & (freq_axis_out <= target_high)
    for d in np.where(in_band)[0]:
        src_lo, src_hi = np.searchsorted(boundaries, [d - 0.5, d + 0.5])
        src_lo = min(src_lo, n_src - 1)
        src_hi = min(max(src_hi, src_lo + 1), n_src)
        remapped[d, :] = spectrogram[src_lo:src_hi, :].mean(axis=0)
    return remapped, a, b


def image_to_magnitude(image_path):
    """Exactly your I-GLA preprocessing + frequency_remap_hires, stopped right
    before Griffin-Lim. Returns only the in-band rows (everything else is 0)."""
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError("Image not found or invalid path.")
    image = cv2.resize(image, (IMG_W, IMG_H))
    image_normalized = np.flipud(image.astype(np.float32) / 255.0)
    spectrogram_db = librosa.amplitude_to_db(image_normalized + 1e-7, ref=np.max)
    spectrogram_linear = librosa.db_to_amplitude(spectrogram_db)
    spectrogram_linear, _, _ = frequency_remap_hires(spectrogram_linear, sr=SR, n_fft_out=N_FFT)
    return spectrogram_linear[BAND_LO:BAND_HI].astype(np.float32)


def load_magnitudes():
    """Builds (once) and caches train/test magnitudes with a fixed-seed split."""
    if os.path.exists(CACHE_PATH):
        d = np.load(CACHE_PATH, allow_pickle=True)
        return d["train"], d["test"], list(d["test_files"])

    files = sorted(os.path.join(r, f) for r, _, fs in os.walk(IMG_ROOT)
                   for f in fs if f.lower().endswith(IMG_EXTS))
    random.Random(SEED).shuffle(files)
    n_test = max(1, int(len(files) * TEST_FRAC))

    def stack(file_list):
        mags, kept = [], []
        for p in tqdm(file_list, desc="Building magnitudes"):
            try:
                mags.append(image_to_magnitude(p)); kept.append(p)
            except Exception as e:
                print(f"\n[skip] {p}: {e}")
        return np.stack(mags), kept

    test, test_files = stack(files[:n_test])
    train, _ = stack(files[n_test:])
    np.savez(CACHE_PATH, train=train, test=test, test_files=np.array(test_files))
    return train, test, test_files


# ---- STFT helpers (one definition shared by PhaseNet AND GLA, so SC is comparable) ----
def _win(device):
    return torch.hann_window(WIN, device=device)

def stft_mag(audio):
    return torch.stft(audio, N_FFT, HOP, WIN, _win(audio.device), center=True,
                      pad_mode="constant", return_complex=True).abs()

def band_to_full(band):
    full = torch.zeros(band.shape[0], N_BINS, band.shape[2], dtype=band.dtype, device=band.device)
    full[:, BAND_LO:BAND_HI] = band
    return full

def istft_band(mag_band, unit_phase):
    spec = band_to_full(mag_band.to(torch.complex64) * unit_phase)
    return torch.istft(spec, N_FFT, HOP, WIN, _win(spec.device), center=True, length=AUDIO_LEN)

def spectral_convergence(audio, mag_band):
    """||S - |STFT(audio)|||_F / ||S||_F over the FULL spectrum (lower = better).
    Energy leaking outside 1-4kHz counts as error too."""
    S = band_to_full(mag_band)
    S_hat = stft_mag(audio)
    return torch.linalg.norm((S - S_hat).flatten(1), dim=1) / torch.linalg.norm(S.flatten(1), dim=1)


# ---- PhaseNet: small U-Net, magnitude in -> unit phase (cos, sin) out ----
def _block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class PhaseNet(nn.Module):
    def __init__(self, base=32, depth=4):
        super().__init__()
        chs = [base * 2 ** i for i in range(depth + 1)]
        self.downs = nn.ModuleList()
        cin = 1
        for c in chs[:-1]:
            self.downs.append(_block(cin, c)); cin = c
        self.mid = _block(chs[-2], chs[-1])
        self.ups, self.up_blocks = nn.ModuleList(), nn.ModuleList()
        for c in reversed(chs[:-1]):
            self.ups.append(nn.ConvTranspose2d(c * 2, c, 2, stride=2))
            self.up_blocks.append(_block(c * 2, c))
        self.head = nn.Conv2d(chs[0], 2, 1)

    def forward(self, x):
        skips = []
        for d in self.downs:
            x = d(x); skips.append(x); x = F.max_pool2d(x, 2)
        x = self.mid(x)
        for up, blk, s in zip(self.ups, self.up_blocks, reversed(skips)):
            x = blk(torch.cat([up(x), s], 1))
        return self.head(x)

def phase_from_model(model, mag_band):
    """(B, N_BAND, T) linear magnitude -> (B, N_BAND, T) complex unit phase."""
    x = (torch.log(mag_band + 1e-5) / 11.5 + 1.0).unsqueeze(1)   # roughly [0, 1]
    x = F.pad(x, (0, 0, 0, PAD_H - N_BAND))
    out = model(x)[:, :, :N_BAND].float()
    re, im = out[:, 0], out[:, 1]
    norm = torch.sqrt(re ** 2 + im ** 2 + 1e-8)
    return torch.complex(re / norm, im / norm)

Writing phasenet_common.py


In [3]:
import time, json
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from phasenet_common import *

EPOCHS = 30
BATCH = 8
LR = 3e-4
dev = torch.device("cuda")
torch.manual_seed(SEED)

train, test, _ = load_magnitudes()
train_t, test_t = torch.from_numpy(train), torch.from_numpy(test)
print(f"train: {len(train_t)}  test: {len(test_t)}  input: {N_BAND} bins x {IMG_W} frames")

model = PhaseNet().to(dev)
opt = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)

def loss_fn(mag_band):
    # self-supervised consistency loss: predict phase -> iSTFT -> STFT -> compare to target magnitude
    audio = istft_band(mag_band, phase_from_model(model, mag_band))
    S, S_hat = band_to_full(mag_band), stft_mag(audio)
    sc = (torch.linalg.norm((S - S_hat).flatten(1), dim=1) / torch.linalg.norm(S.flatten(1), dim=1)).mean()
    logmag = F.l1_loss(torch.log(S_hat + 1e-5), torch.log(S + 1e-5))
    return sc + logmag

@torch.no_grad()
def test_sc():
    model.eval()
    vals = [spectral_convergence(istft_band(m, phase_from_model(model, m)), m)
            for m in (test_t[i:i + BATCH].to(dev) for i in range(0, len(test_t), BATCH))]
    model.train()
    return torch.cat(vals).mean().item()

t0 = time.perf_counter()
for epoch in range(EPOCHS):
    perm = torch.randperm(len(train_t))
    losses = []
    for i in tqdm(range(0, len(perm), BATCH), desc=f"Epoch {epoch + 1}/{EPOCHS}"):
        loss = loss_fn(train_t[perm[i:i + BATCH]].to(dev))
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    sched.step()
    print(f"  train loss {np.mean(losses):.4f} | test SC {test_sc():.4f}")
minutes = (time.perf_counter() - t0) / 60

torch.save(model.state_dict(), "phasenet.pt")
log = {"pairs": len(train_t), "epochs": EPOCHS, "minutes": minutes,
       "loss": "self-supervised STFT consistency: spectral convergence + log-magnitude L1",
       "input": f"magnitude ({N_BAND} in-band bins x {IMG_W} frames, N_FFT={N_FFT}) -> phase"}
json.dump(log, open("phasenet_train_log.json", "w"), indent=2)
print(json.dumps(log, indent=2))

Building magnitudes: 100%|███████████████████████████████████████████████████████████| 360/360 [00:06<00:00, 51.75it/s]


train: 360  test: 40  input: 558 bins x 128 frames


Epoch 1/30: 100%|██████████████████████████████████████████████████████████████████████| 45/45 [00:07<00:00,  5.98it/s]


  train loss 0.3951 | test SC 0.1048


Epoch 2/30: 100%|██████████████████████████████████████████████████████████████████████| 45/45 [00:06<00:00,  7.21it/s]


...

  train loss 0.2032 | test SC 0.0660


Epoch 29/30: 100%|█████████████████████████████████████████████████████████████████████| 45/45 [00:06<00:00,  7.39it/s]


  train loss 0.2030 | test SC 0.0660


Epoch 30/30: 100%|█████████████████████████████████████████████████████████████████████| 45/45 [00:06<00:00,  7.38it/s]


  train loss 0.2031 | test SC 0.0659
{
  "pairs": 360,
  "epochs": 30,
  "minutes": 3.203868845,
  "loss": "self-supervised STFT consistency: spectral convergence + log-magnitude L1",
  "input": "magnitude (558 in-band bins x 128 frames, N_FFT=4096) -> phase"
}


In [1]:
import time, json
import numpy as np
import torch
import librosa
from tqdm import tqdm
from phasenet_common import *

N_ITER = 32
dev = torch.device("cuda")
_, test, _ = load_magnitudes()

def gla_torch(mag_band, n_iter=32, momentum=0.99):
    """Same algorithm as librosa.griffinlim (fast GLA, momentum 0.99, random init), on GPU."""
    S = band_to_full(mag_band).to(torch.complex64)
    win = torch.hann_window(WIN, device=S.device)
    g = torch.Generator(device=S.device).manual_seed(0)
    angles = torch.polar(torch.ones_like(S.real), 2 * np.pi * torch.rand(S.shape, generator=g, device=S.device))
    rebuilt = torch.zeros_like(S)
    for _ in range(n_iter):
        tprev = rebuilt
        inverse = torch.istft(S * angles, N_FFT, HOP, WIN, win, center=True, length=AUDIO_LEN)
        rebuilt = torch.stft(inverse, N_FFT, HOP, WIN, win, center=True, pad_mode="constant", return_complex=True)
        angles = rebuilt - (momentum / (1 + momentum)) * tprev
        angles = angles / (angles.abs() + 1e-16)
    return torch.istft(S * angles, N_FFT, HOP, WIN, win, center=True, length=AUDIO_LEN)

# ---- librosa on CPU (what the I-GLA paper actually used) ----
sc_cpu, t_cpu = [], []
for m in tqdm(test, desc="GLA librosa (CPU)"):
    full = np.zeros((N_BINS, m.shape[1]), np.float32)
    full[BAND_LO:BAND_HI] = m
    t = time.perf_counter()
    y = librosa.griffinlim(full, n_iter=N_ITER, n_fft=N_FFT, hop_length=HOP,
                           win_length=WIN, length=AUDIO_LEN, random_state=0)
    t_cpu.append(time.perf_counter() - t)
    sc_cpu.append(spectral_convergence(torch.from_numpy(y).float()[None].to(dev),
                                       torch.from_numpy(m)[None].to(dev)).item())

# ---- same algorithm on GPU (fair hardware comparison with PhaseNet) ----
sc_gpu, t_gpu = [], []
with torch.no_grad():
    for _ in range(3):   # warmup
        gla_torch(torch.from_numpy(test[0])[None].to(dev), N_ITER)
    for m in tqdm(test, desc="GLA torch (GPU)"):
        mt = torch.from_numpy(m)[None].to(dev)
        torch.cuda.synchronize(); t = time.perf_counter()
        y = gla_torch(mt, N_ITER)
        torch.cuda.synchronize(); t_gpu.append(time.perf_counter() - t)
        sc_gpu.append(spectral_convergence(y, mt).item())

res = {"n_test": len(test), "n_iter": N_ITER, "N_FFT": N_FFT, "HOP": HOP,
       "cpu_librosa": {"spectral_convergence": float(np.mean(sc_cpu)), "s_per_sample": float(np.mean(t_cpu))},
       "gpu_torch":   {"spectral_convergence": float(np.mean(sc_gpu)), "s_per_sample": float(np.mean(t_gpu))}}
json.dump(res, open("gla_baseline.json", "w"), indent=2)
print(json.dumps(res, indent=2))

GLA torch (GPU): 100%|█████████████████████████████████████████████████████████████████| 40/40 [00:01<00:00, 37.94it/s]

{
  "n_test": 40,
  "n_iter": 32,
  "N_FFT": 4096,
  "HOP": 2048,
  "cpu_librosa": {
    "spectral_convergence": 0.038423371594399215,
    "s_per_sample": 0.5317513675000001
  },
  "gpu_torch": {
    "spectral_convergence": 0.03861639527603984,
    "s_per_sample": 0.02538966999999932
  }
}


In [1]:
import time, json, os
import numpy as np
import torch
from fvcore.nn import FlopCountAnalysis
from phasenet_common import *

dev = torch.device("cuda")
model = PhaseNet().to(dev)
model.load_state_dict(torch.load("phasenet.pt", map_location=dev))
model.eval()

_, test, _ = load_magnitudes()
test_t = torch.from_numpy(test)
x = torch.randn(1, 1, PAD_H, IMG_W, device=dev)    # exact network input shape

@torch.no_grad()
def sc_on_test(fp16):
    vals = []
    for i in range(len(test_t)):
        m = test_t[i:i + 1].to(dev)
        with torch.autocast("cuda", dtype=torch.float16, enabled=fp16):
            unit = phase_from_model(model, m)
        vals.append(spectral_convergence(istft_band(m, unit), m).item())   # STFT stays FP32
    return float(np.mean(vals))

@torch.no_grad()
def lat_mem(fp16, end_to_end=False, warm=20, runs=100):
    m = test_t[0:1].to(dev)
    def step():
        with torch.autocast("cuda", dtype=torch.float16, enabled=fp16):
            if not end_to_end:
                return model(x)
            unit = phase_from_model(model, m)
        return istft_band(m, unit)          # end-to-end = network + iSTFT -> waveform
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    for _ in range(warm): step()
    torch.cuda.synchronize(); ts = []
    for _ in range(runs):
        t = time.perf_counter(); step()
        torch.cuda.synchronize(); ts.append((time.perf_counter() - t) * 1000)
    return float(np.mean(ts)), torch.cuda.max_memory_allocated() / 2 ** 20

res = {"params_M": sum(p.numel() for p in model.parameters()) / 1e6}
with torch.no_grad():
    macs = FlopCountAnalysis(model, x).unsupported_ops_warnings(False).total()
res["GMACs"] = macs / 1e9
res["GFLOPs_2xMACs"] = 2 * macs / 1e9
for tag, f in (("fp32", False), ("fp16", True)):
    l, mem = lat_mem(f)
    l_e2e, _ = lat_mem(f, end_to_end=True)
    res[tag] = {"latency_ms": l, "latency_e2e_ms": l_e2e, "peak_mem_MB": mem, "spectral_convergence": sc_on_test(f)}
res["fp16_sc_delta"] = res["fp16"]["spectral_convergence"] - res["fp32"]["spectral_convergence"]
json.dump(res, open("phasenet_characterization.json", "w"), indent=2)
print(json.dumps(res, indent=2))

# ---- filled-in template ----
gla = json.load(open("gla_baseline.json")) if os.path.exists("gla_baseline.json") else None
tr = json.load(open("phasenet_train_log.json")) if os.path.exists("phasenet_train_log.json") else None
print("\nPHASENET (batch 1, RTX 4060)")
print(f"params {res['params_M']:.2f} M | GMACs {res['GMACs']:.2f} | latency FP32 {res['fp32']['latency_ms']:.2f} ms "
      f"| peak mem FP32 {res['fp32']['peak_mem_MB']:.0f} MB")
print(f"FP16: {res['fp16']['latency_ms']:.2f} ms / {res['fp16']['peak_mem_MB']:.0f} MB "
      f"/ spectral-convergence delta {res['fp16_sc_delta']:+.4f}")
if gla:
    print(f"Quality: spectral convergence PhaseNet {res['fp32']['spectral_convergence']:.4f} vs "
          f"GLA({gla['n_iter']} iter) {gla['cpu_librosa']['spectral_convergence']:.4f} (lower = better)")
    print(f"Speed: GLA {gla['cpu_librosa']['s_per_sample']:.2f} s/sample (CPU librosa), "
          f"{gla['gpu_torch']['s_per_sample'] * 1000:.1f} ms/sample (GPU) vs "
          f"PhaseNet {res['fp32']['latency_e2e_ms']:.1f} ms/sample (net + iSTFT)")
if tr:
    print(f"Training: {tr['pairs']} magnitudes, {tr['epochs']} epochs, {tr['minutes']:.1f} min, loss: {tr['loss']}")
    print(f"Input/output: {tr['input']}")

C:\Users\User\AppData\Local\Temp\ipykernel_19416\775774917.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("phasenet.pt", map_location=d

{
  "params_M": 7.765442,
  "GMACs": 13.19886848,
  "GFLOPs_2xMACs": 26.39773696,
  "fp32": {
    "latency_ms": 5.26495300000004,
    "latency_e2e_ms": 5.126046999999971,
    "peak_mem_MB": 87.59326171875,
    "spectral_convergence": 0.06587268700823187
  },
  "fp16": {
    "latency_ms": 4.847774000000201,
    "latency_e2e_ms": 6.193873000000103,
    "peak_mem_MB": 83.64208984375,
    "spectral_convergence": 0.06587673975154758
  },
  "fp16_sc_delta": 4.052743315705043e-06
}

PHASENET (batch 1, RTX 4060)
params 7.77 M | GMACs 13.20 | latency FP32 5.26 ms | peak mem FP32 88 MB
FP16: 4.85 ms / 84 MB / spectral-convergence delta +0.0000
Quality: spectral convergence PhaseNet 0.0659 vs GLA(32 iter) 0.0384 (lower = better)
Speed: GLA 0.53 s/sample (CPU librosa), 25.4 ms/sample (GPU) vs PhaseNet 5.1 ms/sample (net + iSTFT)
Training: 360 magnitudes, 30 epochs, 3.2 min, loss: self-supervised STFT consistency: spectral convergence + log-magnitude L1
Input/output: magnitude (558 in-band bins x 1